# PROSES PEMBUATAN MODEL
## Dataset loading
Pertama kami melakukan data loading dan mapping. Tujuan dari model ini mampu mempridiksi tingkat prioritas mana yang harus dikejar. 
Untuk pertama kami melakukan mapping dengan input seperti ini
```python
mapping = {
    "TAS": "literasi",
    "TOC": "numerasi",
    "TOR": "karakter_sosial",
    "TSC": "karakter_moral",
    "WEL": "wellbeing"
}
```
lalu kami membersihkan data dimana seperti data tidak lengkap atau NaN

In [1]:
import pandas as pd

df = pd.read_csv("kemendikbud_1.csv")

mapping = {
    "TAS": "literasi",
    "TOC": "numerasi",
    "TOR": "karakter_sosial",
    "TSC": "karakter_moral",
    "WEL": "wellbeing"
}

subjects = ["TAS", "TOC", "TOR", "TSC", "WEL"]

df_long = df.melt(
    id_vars=["kd_sekolah"],
    value_vars=subjects,
    var_name="indicator",
    value_name="score"
)

df_long["subject"] = df_long["indicator"].map(mapping)
df_long = df_long.dropna(subset=["score"])

df_long.head()


,kd_sekolah,indicator,score,subject
1,8100010,TAS,50.51,literasi
3,8100010,TAS,43.48,literasi
4,8100010,TAS,44.76,literasi
7,8100010,TAS,46.29,literasi
9,8100010,TAS,62.66,literasi


Lalu kami menambahkan kolom priority sebagai target variable

In [2]:
df_long["priority"] = 100 - df_long["score"]


## Train, test, split
disini kami membagi dataset sebagai train, test. Disini kami memilih Algoritma Neural Network, 
1. karena masalah ini melibatkan multidimensi, atau banyak target variable seperti di mapping diatas. Dan hubungan antara variable yang tidak non linear
2. Output yang berupa multidimensi
3. pola performa siswa yang cukup abstrak
maka kami memilih neural network karena NN mampu mengatasi masalah diatas

### Model architecture
dengan hidden layer sebanyak 2 dan fungsi aktifase RelU juga dengan output model dengan aktivasi 

In [3]:
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split

# ===== Data preparation =====
df_long["subject_id"] = df_long["subject"].astype("category").cat.codes

X = torch.tensor(df_long[["subject_id","score"]].values, dtype=torch.float32)
y = torch.tensor(df_long["priority"].values, dtype=torch.float32).view(-1,1)

# Split train & validation
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# ===== Model definition =====
class Recommender(nn.Module):
    def __init__(self, num_subjects, embed_dim=8):  # embedding lebih besar
        super().__init__()
        self.embed = nn.Embedding(num_subjects, embed_dim)
        self.fc = nn.Sequential(
            nn.Linear(embed_dim + 1, 32),  # hidden layer lebih besar
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()  # output di 0-1
        )

    def forward(self, x):
        subj = x[:,0].long()
        score = x[:,1].unsqueeze(1)
        emb = self.embed(subj)
        out = torch.cat([emb, score], dim=1)
        return self.fc(out)

# ===== Init model, loss, optimizer =====
model = Recommender(df_long["subject_id"].nunique())
loss_fn = nn.MSELoss()
opt = torch.optim.Adam(model.parameters(), lr=0.001)

# ===== Training loop =====
epochs = 600
for epoch in range(epochs):
    # --- Train ---
    model.train()
    pred_train = model(X_train)
    loss_train = loss_fn(pred_train, y_train / 100)
    opt.zero_grad()
    loss_train.backward()
    opt.step()

    # --- Validation ---
    model.eval()
    with torch.no_grad():
        pred_val = model(X_val)
        loss_val = loss_fn(pred_val, y_val / 100)
        error_val = torch.abs(pred_val - (y_val / 100))
        val_acc = (error_val < 0.1).float().mean().item()

    # Print every 50 epochs
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/{epochs} | "
              f"Train Loss: {loss_train.item():.6f} | "
              f"Val Loss: {loss_val.item():.6f} | "
              f"Val Acc: {val_acc*100:.2f}%")

print("Training done.")

# ===== Save model =====
torch.save(model.state_dict(), "recommender_finetuned.pth")
print("Model saved as recommender_finetuned.pth")


Epoch 50/600 | Train Loss: 0.018302 | Val Loss: 0.018004 | Val Acc: 45.67%
Epoch 100/600 | Train Loss: 0.008950 | Val Loss: 0.008818 | Val Acc: 65.92%
Epoch 150/600 | Train Loss: 0.001755 | Val Loss: 0.001695 | Val Acc: 98.79%
Epoch 200/600 | Train Loss: 0.000749 | Val Loss: 0.000750 | Val Acc: 99.99%
Epoch 250/600 | Train Loss: 0.000739 | Val Loss: 0.000741 | Val Acc: 100.00%
Epoch 300/600 | Train Loss: 0.000739 | Val Loss: 0.000740 | Val Acc: 100.00%
Epoch 350/600 | Train Loss: 0.000738 | Val Loss: 0.000740 | Val Acc: 100.00%
Epoch 400/600 | Train Loss: 0.000737 | Val Loss: 0.000739 | Val Acc: 100.00%
Epoch 450/600 | Train Loss: 0.000740 | Val Loss: 0.000740 | Val Acc: 100.00%
Epoch 500/600 | Train Loss: 0.000736 | Val Loss: 0.000738 | Val Acc: 100.00%
Epoch 550/600 | Train Loss: 0.000736 | Val Loss: 0.000737 | Val Acc: 100.00%
Epoch 600/600 | Train Loss: 0.000735 | Val Loss: 0.000744 | Val Acc: 100.00%
Training done.
Model saved as recommender_finetuned.pth


# save model 

In [4]:
dummy_input = torch.randn(1, 2)

torch.onnx.export(
    model,
    dummy_input,
    "recommender.onnx",
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}},
    opset_version=14
)

print("Model exported to recommender.onnx")

Model exported to recommender.onnx
